In [ ]:
import numpy as np
import torch
import timesfm

torch.set_float32_matmul_precision("high")

In [ ]:
import huggingface_hub
huggingface_hub.login()

In [ ]:
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

## Compile

max_context=1024: 과거 데이터 최대 1024개 사용
max_horizon=256: 미래 최대 256개 예측
normalize_inputs=True: 입력 시계열 자동 정규화
infer_is_positive=False: 음수 예측을 허용
use_continuous_quantile_head=True: 불확실성 구간도

In [ ]:
model.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=False,
        fix_quantile_crossing=True,
    )
)



## 예시 1. Zeroshot 예측

In [ ]:
t = np.linspace(0, 30, 512)

sensor_history = (
    np.sin(t)
    + 0.2 * np.sin(3 * t)
    + 0.05 * np.random.randn(512)
).astype(np.float32)

In [ ]:
# TimesFM 입력은 시계열들의 리스트
inputs = [sensor_history]

In [ ]:
point_forecast, quantile_forecast = model.forecast(
    horizon=96,
    inputs=inputs,
)

In [ ]:
print(point_forecast.shape)
print(quantile_forecast.shape)

In [ ]:
forecast = point_forecast[0]

print(forecast.shape)
# (96,)

In [ ]:
import matplotlib.pyplot as plt

history_length = len(sensor_history)
forecast_length = len(forecast)

plt.figure(figsize=(14, 5))

plt.plot(
    np.arange(history_length),
    sensor_history,
    label="Past sensor data"
)

plt.plot(
    np.arange(history_length, history_length + forecast_length),
    forecast,
    label="TimesFM forecast"
)

plt.axvline(
    history_length - 1,
    linestyle="--",
    label="Forecast start"
)

plt.xlabel("Time")
plt.ylabel("Sensor value")
plt.title("TimesFM zero-shot forecasting")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 불확실성도 같이 예측

In [ ]:
q_forecast = quantile_forecast[0]

mean_pred = q_forecast[:, 0]
lower_pred = q_forecast[:, 1]
upper_pred = q_forecast[:, -1]

future_index = np.arange(
    len(sensor_history),
    len(sensor_history) + len(forecast)
)

plt.figure(figsize=(14, 5))

plt.plot(sensor_history, label="History")

plt.plot(
    future_index,
    forecast,
    label="Point forecast"
)

plt.fill_between(
    future_index,
    lower_pred,
    upper_pred,
    alpha=0.2,
    label="Prediction interval"
)

plt.axvline(len(sensor_history) - 1, linestyle="--")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 예시 2

In [ ]:
t = np.linspace(0, 36, 608)

full_series = (
    np.sin(t)
    + 0.2 * np.sin(3 * t)
    + 0.05 * np.random.randn(608)
).astype(np.float32)

history = full_series[:512]
actual_future = full_series[512:]

In [ ]:
point_forecast, quantile_forecast = model.forecast(
    horizon=len(actual_future),
    inputs=[history],
)

prediction = point_forecast[0]

In [ ]:
mae = np.mean(np.abs(actual_future - prediction))
rmse = np.sqrt(np.mean((actual_future - prediction) ** 2))

print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    np.arange(len(history)),
    history,
    label="History"
)

future_index = np.arange(
    len(history),
    len(history) + len(actual_future)
)

plt.plot(
    future_index,
    actual_future,
    label="Actual future"
)

plt.plot(
    future_index,
    prediction,
    label="TimesFM prediction"
)

plt.axvline(len(history) - 1, linestyle="--")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 여러 센서 입력 가능

단 기본적으로 독립된 하나의 시계열로 처리

In [ ]:
sensor_1 = np.sin(np.linspace(0, 20, 300)).astype(np.float32)
sensor_2 = np.cos(np.linspace(0, 15, 450)).astype(np.float32)
sensor_3 = np.linspace(0, 1, 200).astype(np.float32)

inputs = [
    sensor_1,
    sensor_2,
    sensor_3,
]

sensor_names = [
    "Sensor 1: Sine",
    "Sensor 2: Cosine",
    "Sensor 3: Linear trend",
]

In [ ]:
sensor_1.shape, sensor_2.shape, sensor_3.shape

In [ ]:
fig, axes = plt.subplots(
    nrows=len(inputs),
    ncols=1,
    figsize=(14, 8),
)

for i, (sensor, name) in enumerate(zip(inputs, sensor_names)):
    axes[i].plot(
        np.arange(len(sensor)),
        sensor,
    )

    axes[i].set_title(
        f"{name} — Length: {len(sensor)}"
    )
    axes[i].set_ylabel("Value")
    axes[i].grid(alpha=0.3)

axes[-1].set_xlabel("Time step")

plt.suptitle(
    "Input time series for TimesFM",
    fontsize=15,
)

plt.tight_layout()
plt.show()

In [ ]:
horizon=50
point_forecast, quantile_forecast = model.forecast(
    horizon=horizon,
    inputs=inputs,
)

print(point_forecast.shape)
# (3, 24)

In [ ]:
fig, axes = plt.subplots(
    nrows=len(inputs),
    ncols=1,
    figsize=(14, 9),
)

for i, (history, name) in enumerate(zip(inputs, sensor_names)):
    prediction = point_forecast[i]

    history_index = np.arange(len(history))
    forecast_index = np.arange(
        len(history),
        len(history) + horizon,
    )

    axes[i].plot(
        history_index,
        history,
        label="History",
    )

    axes[i].plot(
        forecast_index,
        prediction,
        label="TimesFM forecast",
    )

    axes[i].axvline(
        len(history) - 1,
        linestyle="--",
        label="Forecast start",
    )

    axes[i].set_title(
        f"{name} — Context length: {len(history)}"
    )
    axes[i].set_ylabel("Value")
    axes[i].grid(alpha=0.3)
    axes[i].legend()

axes[-1].set_xlabel("Time step")

plt.suptitle(
    f"TimesFM zero-shot forecasting — Horizon: {horizon}",
    fontsize=15,
)

plt.tight_layout()
plt.show()